In [2]:
import pandas as pd
import numpy as np
import os
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
from scipy.stats import shapiro
import matplotlib.pyplot as plt
import seaborn as sns


In [4]:
path_data_clean = "../data_clean/"

file_data_bersih = os.path.join(path_data_clean, "dataset_final_2021-2023.csv")


df = pd.read_csv(file_data_bersih)
df.head()

,Wilayah,Tahun,P1,UHH,HLS,RLS,Pengeluaran,TPT,Kepadatan
0,Kabupaten Cilacap,2021,1.48,73.90,12.63,7.09,10534,9.97,924
1,Kabupaten Banyumas,2021,2.35,73.80,13.03,7.63,11546,6.05,1340
2,Kabupaten Purbalingga,2021,2.10,73.21,12.00,7.25,10032,6.05,1487
3,Kabupaten Banjarnegara,2021,2.97,74.28,11.63,6.75,9407,5.86,1003
4,Kabupaten Kebumen,2021,3.24,73.55,13.35,7.55,9028,6.03,1124


In [5]:
print(df.shape)

(105, 9)


Data terdiri dari 105 baris, 8 kolom

In [6]:
print("Jumlah data duplikat: ", df.duplicated().sum())

Jumlah data duplikat:  0


In [7]:
print(df.isna().sum())

Wilayah        0
Tahun          0
P1             0
UHH            0
HLS            0
RLS            0
Pengeluaran    0
TPT            0
Kepadatan      0
dtype: int64


In [8]:
print(df.describe())

             Tahun          P1         UHH         HLS         RLS  \
count   105.000000  105.000000  105.000000  105.000000  105.000000   
mean   2022.000000    1.618000   75.119333   13.012667    8.125048   
std       0.820413    0.691389    1.807124    0.912235    1.267850   
min    2021.000000    0.470000   69.540000   11.630000    6.220000   
25%    2021.000000    1.030000   74.030000   12.440000    7.260000   
50%    2022.000000    1.570000   74.870000   12.890000    7.790000   
75%    2023.000000    1.950000   76.390000   13.350000    8.790000   
max    2023.000000    3.410000   77.930000   15.550000   11.240000   

        Pengeluaran         TPT     Kepadatan  
count    105.000000  105.000000    105.000000  
mean   11565.742857    5.361429   2113.942857  
std     1814.000631    1.947240   2413.246293  
min     8573.000000    1.760000    461.000000  
25%    10277.000000    4.050000    930.000000  
50%    11110.000000    5.030000   1180.000000  
75%    12522.000000    6.350000  

PENGERJAAN

STEP 1 — Pisahkan data
- Train = tahun 2021 & 2022
- Test = tahun 2023

In [9]:
df_train = df[df["Tahun"].isin([2021, 2022])].copy()
df_test  = df[df["Tahun"] == 2023].copy()

print(df_train.shape, df_test.shape)

(70, 9) (35, 9)


STEP 2 — Uji signifikansi sederhana (satu-satu hubungan)

Regress:
- P1 ~ UHH
- P1 ~ HLS
- P1 ~ RLS
- P1 ~ Pengeluaran
- P1 ~ TPT
- P1 ~ Kepadatan

Tujuan uji ini: melihat variabel mana yang memiliki hubungan jelas dengan Y.

In [10]:
def simple_regression(df, x):
    X = sm.add_constant(df[[x]])
    y = df["P1"]
    model = sm.OLS(y, X).fit()
    print(f"\n===== P1 ~ {x} =====")
    print(model.summary())

for col in ["UHH", "HLS", "RLS", "Pengeluaran", "TPT", "Kepadatan"]:
    simple_regression(df_train, col)


===== P1 ~ UHH =====
                            OLS Regression Results                            
Dep. Variable:                     P1   R-squared:                       0.344
Model:                            OLS   Adj. R-squared:                  0.334
Method:                 Least Squares   F-statistic:                     35.65
Date:                Fri, 28 Nov 2025   Prob (F-statistic):           9.56e-08
Time:                        15:56:55   Log-Likelihood:                -59.435
No. Observations:                  70   AIC:                             122.9
Df Residuals:                      68   BIC:                             127.4
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         18.7384      2.8

STEP 3 — Bangun model regresi linier berganda (X1…X6)

Lakukan:
- uji F
- uji t
- cek VIF
- backward elimination kalau ada variabel tidak signifikan
- pilih model final

In [11]:
X_cols = ["UHH","HLS","RLS","Pengeluaran","TPT","Kepadatan"]
X_train = sm.add_constant(df_train[X_cols])
y_train = df_train["P1"]

model_full = sm.OLS(y_train, X_train).fit()
print(model_full.summary())


                            OLS Regression Results                            
Dep. Variable:                     P1   R-squared:                       0.436
Model:                            OLS   Adj. R-squared:                  0.383
Method:                 Least Squares   F-statistic:                     8.124
Date:                Fri, 28 Nov 2025   Prob (F-statistic):           1.62e-06
Time:                        15:57:33   Log-Likelihood:                -54.130
No. Observations:                  70   AIC:                             122.3
Df Residuals:                      63   BIC:                             138.0
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const          13.6849      4.852      2.820      

In [13]:
# cek uji multikolinearitas (VIF)
from statsmodels.stats.outliers_influence import variance_inflation_factor
import pandas as pd

vif_df = pd.DataFrame()
vif_df["feature"] = X_cols
vif_df["VIF"] = [variance_inflation_factor(df_train[X_cols].values, i)
                 for i in range(len(X_cols))]

vif_df


,feature,VIF
0,UHH,620.245192
1,HLS,1877.487423
2,RLS,602.106523
3,Pengeluaran,143.829491
4,TPT,16.290746
5,Kepadatan,6.254313


In [14]:
# Backward elimination jika ada variabel tidak signifikan
def backward_elimination(X, y, alpha=0.05):
    X = sm.add_constant(X)
    while True:
        model = sm.OLS(y, X).fit()
        pvals = model.pvalues.drop("const")
        max_p = pvals.max()
        if max_p > alpha:
            drop_var = pvals.idxmax()
            print("Drop:", drop_var, "(p =", max_p, ")")
            X = X.drop(columns=[drop_var])
        else:
            break
    return model, X.columns

model_final, selected_features = backward_elimination(df_train[X_cols], y_train)
print(selected_features)
print(model_final.summary())


Drop: Pengeluaran (p = 0.9481303121317743 )
Drop: HLS (p = 0.45650069358910317 )
Drop: Kepadatan (p = 0.6487857729102886 )
Drop: TPT (p = 0.30751396758945326 )
Index(['const', 'UHH', 'RLS'], dtype='object')
                            OLS Regression Results                            
Dep. Variable:                     P1   R-squared:                       0.420
Model:                            OLS   Adj. R-squared:                  0.403
Method:                 Least Squares   F-statistic:                     24.28
Date:                Fri, 28 Nov 2025   Prob (F-statistic):           1.17e-08
Time:                        15:58:59   Log-Likelihood:                -55.107
No. Observations:                  70   AIC:                             116.2
Df Residuals:                      67   BIC:                             123.0
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
   

STEP 4 — Prediksi tahun 2023 (test set)

- Ambil model final dari train (2021–2022)
- Prediksi P1 tahun 2023
- Hitung MAE, RMSE, MAPE

In [15]:
use_cols = [c for c in selected_features if c != "const"]

X_test = sm.add_constant(df_test[use_cols])
df_test["P1_pred"] = model_final.predict(X_test)

df_test[["Wilayah", "Tahun", "P1", "P1_pred"]]


,Wilayah,Tahun,P1,P1_pred
70,Kabupaten Cilacap,2023,1.54,1.904130
71,Kabupaten Banyumas,2023,1.78,1.826345
72,Kabupaten Purbalingga,2023,2.52,2.015824
73,Kabupaten Banjarnegara,2023,2.34,1.998929
74,Kabupaten Kebumen,2023,2.89,1.845718
75,Kabupaten Purworejo,2023,1.78,1.552569
76,Kabupaten Wonosobo,2023,2.60,2.254508
77,Kabupaten Magelang,2023,1.73,1.812560
78,Kabupaten Boyolali,2023,1.02,1.519915
79,Kabupaten Klaten,2023,1.70,1.157159


In [16]:
y_true = df_test["P1"]
y_pred = df_test["P1_pred"]

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

print("MAE:", mae)
print("RMSE:", rmse)
print("MAPE:", mape)


MAE: 0.4085508622863201
RMSE: 0.4969237130580759
MAPE: 32.0769482637246
